# BernardUdo Sidekick

Self-contained notebook (no separate `.py` files).

Set `OPENROUTER_API_KEY` or `OPENAI_API_KEY` in `.env`. Optional: `SERPER_API_KEY`, `PLAYWRIGHT_HEADLESS=false`.

`cd` to this folder (or a parent) so `.env` and `sandbox/` resolve. Run once: `uv run playwright install chromium`.
On Windows in Jupyter, Playwright uses sync browser + a worker thread (see the tools cell below).


## 1. Path


In [ ]:
from __future__ import annotations

from pathlib import Path

_ROOT = Path.cwd().resolve()
print("ROOT =", _ROOT)


## 2. Environment


In [ ]:
from __future__ import annotations

from pathlib import Path

from dotenv import load_dotenv


def load_project_dotenv() -> None:
    candidates: list[Path] = []

    for p in (Path.cwd(), *Path.cwd().parents):
        candidates.append(p / ".env")
        if p == p.parent:
            break

    for p in (_ROOT, *_ROOT.parents):
        candidates.append(p / ".env")
        if p == p.parent:
            break

    seen: set[Path] = set()
    for env_path in candidates:
        if env_path in seen:
            continue
        seen.add(env_path)
        if env_path.is_file():
            load_dotenv(env_path, override=True)
            return

    load_dotenv(override=True)


## 3. Sidekick tools


In [ ]:
from __future__ import annotations

import asyncio
import concurrent.futures
import functools
import os
import sys
import types
from pathlib import Path
from typing import Any

import requests
from langchain.agents import Tool
from langchain_community.agent_toolkits import FileManagementToolkit, PlayWrightBrowserToolkit
from langchain_community.tools.wikipedia.tool import WikipediaQueryRun
from langchain_community.utilities.wikipedia import WikipediaAPIWrapper
from langchain_experimental.tools import PythonREPLTool

if sys.platform == "win32":
    try:
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    except RuntimeError:
        pass

load_project_dotenv()

_PW_WIN_EXECUTOR: concurrent.futures.ThreadPoolExecutor | None = None

PUSHOVER_URL = "https://api.pushover.net/1/messages.json"
_serper: object | None = None


def _get_pw_executor() -> concurrent.futures.ThreadPoolExecutor:
    global _PW_WIN_EXECUTOR
    if _PW_WIN_EXECUTOR is None:
        _PW_WIN_EXECUTOR = concurrent.futures.ThreadPoolExecutor(
            max_workers=1, thread_name_prefix="playwright-sync"
        )
    return _PW_WIN_EXECUTOR


def _get_serper():
    global _serper
    if _serper is None and os.getenv("SERPER_API_KEY"):
        from langchain_community.utilities import GoogleSerperAPIWrapper

        _serper = GoogleSerperAPIWrapper()
    return _serper


def _search(query: str) -> str:
    s = _get_serper()
    if s is None:
        return (
            "Web search is unavailable: set SERPER_API_KEY in your environment "
            "(https://serper.dev). You can still use browser tools with a URL or Wikipedia."
        )
    return s.run(query)


def push(text: str) -> str:
    token = os.getenv("PUSHOVER_TOKEN")
    user = os.getenv("PUSHOVER_USER")
    if not token or not user:
        return "Pushover not configured (set PUSHOVER_TOKEN and PUSHOVER_USER)."
    requests.post(PUSHOVER_URL, data={"token": token, "user": user, "message": text}, timeout=30)
    return "Notification sent."


def _sandbox_dir() -> Path:
    d = _ROOT / "sandbox"
    d.mkdir(exist_ok=True)
    return d


def get_file_tools():
    toolkit = FileManagementToolkit(root_dir=str(_sandbox_dir()))
    return toolkit.get_tools()


def save_file(content: str, filename: str) -> str:
    path = _sandbox_dir() / filename
    try:
        path.write_text(content, encoding="utf-8")
        return f"Successfully saved: {path}"
    except OSError as e:
        return f"Error saving file: {e}"


def _playwright_tools_windows_sync(headless: bool) -> tuple[list, Any]:
    import asyncio.events as _aio_events

    from langchain_community.tools.playwright.utils import create_sync_playwright_browser

    if sys.platform == "win32":
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

    executor = _get_pw_executor()

    def _pw_on_worker_hide_first_running_loop(fn):
        real_grl = asyncio.get_running_loop
        state = {"n": 0}

        def _shim():
            state["n"] += 1
            if state["n"] == 1:
                raise RuntimeError("no running event loop")
            return real_grl()

        asyncio.get_running_loop = _shim  # type: ignore[method-assign]
        try:
            return fn()
        finally:
            asyncio.get_running_loop = real_grl  # type: ignore[method-assign]

    def _make_sync_browser():
        _aio_events._set_running_loop(None)
        return _pw_on_worker_hide_first_running_loop(
            lambda: create_sync_playwright_browser(headless=headless)
        )

    sync_browser = _make_sync_browser()
    toolkit = PlayWrightBrowserToolkit.from_browser(sync_browser=sync_browser)
    tools = toolkit.get_tools()

    def _pw_pin_tool_run_to_playwright_worker(t):
        if getattr(t, "_pw_win_run_pinned", False):
            return
        bound = t._run

        def _run_on_worker(self, *args, **kwargs):
            def inner():
                return bound(*args, **kwargs)

            return executor.submit(inner).result()

        t._run = types.MethodType(_run_on_worker, t)
        t._pw_win_run_pinned = True

    for _t in tools:
        if getattr(_t, "sync_browser", None) is not None:
            _pw_pin_tool_run_to_playwright_worker(_t)

    return tools, sync_browser


async def playwright_tools():
    headless = os.getenv("PLAYWRIGHT_HEADLESS", "true").lower() in ("1", "true", "yes")

    if sys.platform == "win32":
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        loop = asyncio.get_running_loop()
        ex = _get_pw_executor()
        tools, browser = await loop.run_in_executor(
            ex,
            functools.partial(_playwright_tools_windows_sync, headless),
        )
        return tools, browser, None

    from playwright.async_api import async_playwright

    pw = await async_playwright().start()
    browser = await pw.chromium.launch(headless=headless)
    toolkit = PlayWrightBrowserToolkit.from_browser(async_browser=browser)
    return toolkit.get_tools(), browser, pw


async def other_tools() -> list:
    push_tool = Tool(
        name="send_push_notification",
        func=push,
        description="Send a Pushover push notification to the user (optional).",
    )
    save_file_tool = Tool(
        name="save_file",
        func=save_file,
        description=(
            "Save content to a file in the sandbox directory when the user asks to create or write a file. "
            "Arguments: content string, then filename (e.g. report.md, page.html)."
        ),
    )
    search_tool = Tool(
        name="search",
        func=_search,
        description=(
            "Google web search via Serper (requires SERPER_API_KEY). "
            "Use for current events, places, or when you need URLs before browsing."
        ),
    )
    wikipedia = WikipediaAPIWrapper()
    wiki_tool = WikipediaQueryRun(api_wrapper=wikipedia)
    python_repl = PythonREPLTool()
    return get_file_tools() + [push_tool, save_file_tool, search_tool, python_repl, wiki_tool]


## 4. Sidekick


In [ ]:
from __future__ import annotations

import asyncio
import inspect
import os
import sys
import uuid
from datetime import datetime
from typing import Annotated, Any, Dict, List, Optional

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from pydantic import BaseModel, Field
from typing_extensions import TypedDict


class EvaluatorOutput(BaseModel):
    feedback: str = Field(description="Feedback on the assistant's response")
    success_criteria_met: bool = Field(description="Whether the success criteria have been met")
    user_input_needed: bool = Field(
        description="True if more input is needed from the user, or clarifications, or the assistant is stuck"
    )


class State(TypedDict):
    messages: Annotated[List[Any], add_messages]
    success_criteria: str
    feedback_on_work: Optional[str]
    success_criteria_met: bool
    user_input_needed: bool


def make_llm(model: str | None = None) -> ChatOpenAI:
    api_key = os.getenv("OPENROUTER_API_KEY") or os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise ValueError(
            "Set OPENROUTER_API_KEY or OPENAI_API_KEY in your environment or .env file."
        )
    if os.getenv("OPENROUTER_API_KEY"):
        resolved = model or os.getenv("OPENROUTER_MODEL", "openai/gpt-4o-mini")
        base_url = os.getenv("OPENROUTER_API_BASE", "https://openrouter.ai/api/v1")
        return ChatOpenAI(
            model=resolved,
            api_key=api_key,
            base_url=base_url,
            default_headers={
                "HTTP-Referer": os.getenv("OPENROUTER_HTTP_REFERER", "http://localhost"),
                "X-Title": os.getenv("OPENROUTER_APP_TITLE", "Sidekick"),
            },
        )
    resolved = model or os.getenv("OPENAI_MODEL", "gpt-4o-mini")
    return ChatOpenAI(model=resolved, api_key=api_key)


class Sidekick:
    def __init__(self) -> None:
        self.worker_llm_with_tools = None
        self.evaluator_llm_with_output = None
        self.tools: list | None = None
        self.graph = None
        self.sidekick_id = str(uuid.uuid4())
        self.memory = MemorySaver()
        self.browser = None
        self.playwright = None

    async def setup(self) -> None:
        self.tools, self.browser, self.playwright = await playwright_tools()
        self.tools += await other_tools()
        worker_llm = make_llm()
        self.worker_llm_with_tools = worker_llm.bind_tools(self.tools)
        evaluator_llm = make_llm()
        self.evaluator_llm_with_output = evaluator_llm.with_structured_output(EvaluatorOutput)
        await self.build_graph()

    def worker(self, state: State) -> Dict[str, Any]:
        system_message = f"""You are a helpful assistant that can use tools to complete tasks.
You keep working on a task until either you have a question or clarification for the user, or the success criteria is met.
You have tools to browse the web, search (if configured), run Python (use print() for output), manage files in the sandbox, save files with save_file, and use Wikipedia.
The current date and time is {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

This is the success criteria:
{state["success_criteria"]}
You should reply either with a question for the user about this assignment, or with your final response.
If you have a question for the user, state it clearly, for example:

Question: please clarify whether you want a summary or a detailed answer

If you've finished, reply with the final answer only (no question).
"""

        if state.get("feedback_on_work"):
            system_message += f"""
Previously you thought you completed the assignment, but your reply was rejected because the success criteria was not met.
Here is the feedback on why this was rejected:
{state["feedback_on_work"]}
With this feedback, please continue the assignment, ensuring that you meet the success criteria or have a question for the user."""

        found_system_message = False
        messages = state["messages"]
        for message in messages:
            if isinstance(message, SystemMessage):
                message.content = system_message
                found_system_message = True

        if not found_system_message:
            messages = [SystemMessage(content=system_message)] + messages

        response = self.worker_llm_with_tools.invoke(messages)
        return {"messages": [response]}

    def worker_router(self, state: State) -> str:
        last_message = state["messages"][-1]
        if hasattr(last_message, "tool_calls") and last_message.tool_calls:
            return "tools"
        return "evaluator"

    def format_conversation(self, messages: List[Any]) -> str:
        conversation = "Conversation history:\n\n"
        for message in messages:
            if isinstance(message, HumanMessage):
                conversation += f"User: {message.content}\n"
            elif isinstance(message, AIMessage):
                text = message.content or "[Tool calls]"
                if hasattr(message, "tool_calls") and message.tool_calls:
                    names = [
                        tc.get("name", "unknown") if isinstance(tc, dict) else getattr(tc, "name", "unknown")
                        for tc in message.tool_calls
                    ]
                    conversation += f"Assistant: [Used tools: {', '.join(names)}] {text}\n"
                else:
                    conversation += f"Assistant: {text}\n"
        return conversation

    def evaluator(self, state: State) -> State:
        last_msg = state["messages"][-1]
        last_response = getattr(last_msg, "content", None) or ""

        system_message = """You are an evaluator that determines if a task has been completed successfully by an Assistant.
Assess the Assistant's last response based on the given criteria. Respond with your feedback, and with your decision on whether the success criteria has been met,
and whether more input is needed from the user."""

        user_message = f"""You are evaluating a conversation between the User and Assistant.

The entire conversation:
{self.format_conversation(state["messages"])}

The success criteria for this assignment is:
{state["success_criteria"]}

The final response from the Assistant you are evaluating is:
{last_response}

Decide if the success criteria is met by this response.
Decide if more user input is required (question, clarification, or stuck).
"""
        if state.get("feedback_on_work"):
            user_message += (
                f"\nPrior feedback you gave: {state['feedback_on_work']}\n"
                "If the Assistant keeps repeating mistakes, set user_input_needed to true."
            )

        evaluator_messages = [
            SystemMessage(content=system_message),
            HumanMessage(content=user_message),
        ]
        eval_result = self.evaluator_llm_with_output.invoke(evaluator_messages)
        return {
            "messages": [
                {
                    "role": "assistant",
                    "content": f"Evaluator Feedback on this answer: {eval_result.feedback}",
                }
            ],
            "feedback_on_work": eval_result.feedback,
            "success_criteria_met": eval_result.success_criteria_met,
            "user_input_needed": eval_result.user_input_needed,
        }

    def route_based_on_evaluation(self, state: State) -> str:
        if state["success_criteria_met"] or state["user_input_needed"]:
            return "END"
        return "worker"

    async def build_graph(self) -> None:
        graph_builder = StateGraph(State)
        graph_builder.add_node("worker", self.worker)
        graph_builder.add_node("tools", ToolNode(tools=self.tools))
        graph_builder.add_node("evaluator", self.evaluator)
        graph_builder.add_conditional_edges(
            "worker",
            self.worker_router,
            {"tools": "tools", "evaluator": "evaluator"},
        )
        graph_builder.add_edge("tools", "worker")
        graph_builder.add_conditional_edges(
            "evaluator",
            self.route_based_on_evaluation,
            {"worker": "worker", "END": END},
        )
        graph_builder.add_edge(START, "worker")
        self.graph = graph_builder.compile(checkpointer=self.memory)

    async def run_superstep(
        self,
        message: str | List[Any],
        success_criteria: str,
        history: List[dict] | None,
    ) -> List[dict]:
        if history is None:
            history = []
        config = {"configurable": {"thread_id": self.sidekick_id}, "recursion_limit": 50}

        user_messages: List[Any]
        if isinstance(message, str):
            user_messages = [HumanMessage(content=message)]
        else:
            user_messages = message

        state = {
            "messages": user_messages,
            "success_criteria": success_criteria or "The answer should be clear and accurate.",
            "feedback_on_work": None,
            "success_criteria_met": False,
            "user_input_needed": False,
        }
        result = await self.graph.ainvoke(state, config=config)
        msgs = result["messages"]
        last_user = message if isinstance(message, str) else str(message)
        if len(msgs) < 2:
            last_ai = getattr(msgs[-1], "content", None) or ""
            return history + [{"role": "user", "content": last_user}, {"role": "assistant", "content": last_ai}]
        reply = {"role": "assistant", "content": getattr(msgs[-2], "content", None) or ""}
        feedback = {"role": "assistant", "content": getattr(msgs[-1], "content", None) or ""}
        return history + [{"role": "user", "content": last_user}, reply, feedback]

    def cleanup(self) -> None:
        if not self.browser and not self.playwright:
            return
        browser, pw = self.browser, self.playwright
        self.browser, self.playwright = None, None

        async def _close() -> None:
            if browser:
                close_fn = getattr(browser, "close", None)
                if close_fn is not None:
                    if inspect.iscoroutinefunction(close_fn):
                        await close_fn()
                    else:
                        loop = asyncio.get_running_loop()
                        ex = globals().get("_PW_WIN_EXECUTOR")
                        if ex is not None and sys.platform == "win32":
                            await loop.run_in_executor(ex, lambda: close_fn())
                        else:
                            await loop.run_in_executor(None, lambda: close_fn())
            if pw:
                await pw.stop()

        try:
            asyncio.get_running_loop()
        except RuntimeError:
            asyncio.run(_close())
        else:
            asyncio.create_task(_close())


## 5. Gradio app


In [ ]:
import gradio as gr

async def setup() -> Sidekick:
    sidekick = Sidekick()
    await sidekick.setup()
    return sidekick


async def process_message(
    sidekick: Sidekick,
    message: str,
    success_criteria: str,
    history: list | None,
) -> tuple[list, Sidekick]:
    if history is None:
        history = []
    results = await sidekick.run_superstep(message, success_criteria, history)
    return results, sidekick


async def reset() -> tuple[str, str, None, Sidekick]:
    new_sidekick = Sidekick()
    await new_sidekick.setup()
    return "", "", None, new_sidekick


def free_resources(sidekick: Sidekick | None) -> None:
    if sidekick:
        try:
            sidekick.cleanup()
        except Exception as e:
            print(f"Cleanup: {e}")


with gr.Blocks(title="Sidekick Assignment", theme=gr.themes.Default(primary_hue="emerald")) as ui:
    gr.Markdown("## Sidekick - Personal co-worker (Week 4 assignment)")
    gr.Markdown(
        "Set **OPENROUTER_API_KEY** or **OPENAI_API_KEY** in `.env`. "
        "Optional: **SERPER_API_KEY** for Google search, **PLAYWRIGHT_HEADLESS=false** for a visible browser."
    )
    sidekick_state = gr.State(delete_callback=free_resources)

    with gr.Row():
        chatbot = gr.Chatbot(label="Sidekick", height=360, type="messages")
    with gr.Group():
        with gr.Row():
            message = gr.Textbox(show_label=False, placeholder="Your request to your sidekick")
        with gr.Row():
            success_criteria = gr.Textbox(
                show_label=False,
                placeholder='Success criteria (what "done" looks like)',
            )
    with gr.Row():
        reset_button = gr.Button("Reset", variant="stop")
        go_button = gr.Button("Go!", variant="primary")

    ui.load(setup, [], [sidekick_state])
    message.submit(
        process_message,
        [sidekick_state, message, success_criteria, chatbot],
        [chatbot, sidekick_state],
    )
    success_criteria.submit(
        process_message,
        [sidekick_state, message, success_criteria, chatbot],
        [chatbot, sidekick_state],
    )
    go_button.click(
        process_message,
        [sidekick_state, message, success_criteria, chatbot],
        [chatbot, sidekick_state],
    )
    reset_button.click(reset, [], [message, success_criteria, chatbot, sidekick_state])


if __name__ == "__main__":
    ui.launch(inbrowser=True)


## 6. Build Sidekick

Run once: `uv run playwright install chromium`.


In [ ]:
import asyncio

async def build_sidekick() -> Sidekick:
    sk = Sidekick()
    await sk.setup()
    return sk


sidekick = await build_sidekick()
print(type(sidekick.graph))


## 7. Graph (PNG needs Graphviz on PATH)


In [ ]:
from IPython.display import Image, display

g = sidekick.graph.get_graph()
try:
    display(Image(g.draw_mermaid_png()))
except Exception as e:
    print(e)
    print(g.draw_mermaid())


## 8. Demo turn


In [ ]:
history: list = []
out = await sidekick.run_superstep(
    "Reply with the single word: OK",
    "Assistant ends with the word OK.",
    history,
)
for msg in out:
    print(msg.get("role"), ":", (msg.get("content") or "")[:200])


## 9. Gradio inline

LLM calls need network. `cleanup()` avoids two Playwright instances.


In [ ]:
import os

os.environ.setdefault("GRADIO_ANALYTICS_ENABLED", "0")

try:
    sidekick.cleanup()
except NameError:
    pass

ui.launch(inline=True, server_name="127.0.0.1")


## 10. Run as script

Use **Run all** in this notebook, or extract cells into a module if you prefer a CLI.
